# Week 2 — Logistic Regression from Scratch (NumPy)

**Goal:** Understand binary classification by implementing logistic regression yourself — no sklearn for the core model.

We'll:
1. Visualize the **sigmoid**
2. Generate synthetic 2D blobs (two classes)
3. Define **binary cross-entropy (BCE)** and gradients
4. Train with **batch gradient descent**
5. Plot the **decision boundary** and loss curve

## Intuition

A linear score \(z = w \cdot x + b\) is mapped to a probability with the **sigmoid**:

$$\sigma(z) = \frac{1}{1 + e^{-z}} \in (0, 1)$$

We interpret \(p = \sigma(z)\) as \(P(y=1 \mid x)\). Hard labels use a threshold (usually 0.5).

**Binary cross-entropy** measures how well probabilities match labels:

$$L = -\frac{1}{n}\sum_{i=1}^{n}\Big[ y_i \log p_i + (1-y_i)\log(1-p_i) \Big]$$

Gradient descent updates \(w\) and \(b\) to reduce \(L\). For mean BCE the gradients simplify to:

$$\frac{\partial L}{\partial w} = \frac{1}{n} X^\top (p - y), \quad
\frac{\partial L}{\partial b} = \frac{1}{n}\sum_i (p_i - y_i)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Sigmoid visualization

The sigmoid is smooth, S-shaped, and saturates near 0 and 1 for large |z|.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500.0, 500.0)
    return 1.0 / (1.0 + np.exp(-z))

zs = np.linspace(-10, 10, 400)
plt.figure(figsize=(7, 3.5))
plt.plot(zs, sigmoid(zs), color="C0", lw=2)
plt.axhline(0.5, color="gray", ls="--", lw=1)
plt.axvline(0.0, color="gray", ls="--", lw=1)
plt.xlabel("z (logit)")
plt.ylabel("σ(z)")
plt.title("Sigmoid activation")
plt.grid(True, alpha=0.3)
plt.show()

## 2. Synthetic 2D binary data

Two Gaussian blobs — class 0 around (-1.5, -1), class 1 around (1.5, 1). Fully offline; no downloads.

In [ ]:
def make_blobs(n_per_class=100, seed=42):
    rng = np.random.default_rng(seed)
    mean0 = np.array([-1.5, -1.0])
    mean1 = np.array([1.5, 1.0])
    cov = np.array([[0.6, 0.1], [0.1, 0.6]])
    x0 = rng.multivariate_normal(mean0, cov, size=n_per_class)
    x1 = rng.multivariate_normal(mean1, cov, size=n_per_class)
    X = np.vstack([x0, x1])
    y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)])
    perm = rng.permutation(len(y))
    return X[perm], y[perm]

X, y = make_blobs(n_per_class=100, seed=42)

plt.figure(figsize=(5.5, 4.5))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c="C0", edgecolors="k", linewidths=0.4, label="class 0")
plt.scatter(X[y == 1, 0], X[y == 1, 1], c="C3", edgecolors="k", linewidths=0.4, label="class 1")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Synthetic blobs")
plt.legend()
plt.show()
print(f"X shape={X.shape}, y shape={y.shape}, class balance={y.mean():.2f}")

## 3. BCE loss helper

In [ ]:
def binary_cross_entropy(y_true, y_prob, eps=1e-15):
    p = np.clip(y_prob, eps, 1.0 - eps)
    return float(-np.mean(y_true * np.log(p) + (1.0 - y_true) * np.log(1.0 - p)))

# Sanity: predicting 0.5 everywhere on balanced labels ≈ ln(2) ≈ 0.693
dummy = np.full_like(y, 0.5, dtype=float)
print(f"BCE at p=0.5: {binary_cross_entropy(y, dummy):.4f}  (expect ~0.693)")

## 4. LogisticRegression class — fit / predict_proba / predict

In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.1, epochs=300):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0.0
        self.loss_history = []

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        self.loss_history = []
        for _ in range(self.epochs):
            probs = sigmoid(X @ self.w + self.b)
            err = probs - y
            dw = (X.T @ err) / n
            db = float(np.mean(err))
            self.w -= self.lr * dw
            self.b -= self.lr * db
            self.loss_history.append(binary_cross_entropy(y, probs))
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(float)

model = LogisticRegression(lr=0.5, epochs=400)
model.fit(X, y)

preds = model.predict(X)
acc = float(np.mean(y == preds))
print(f"w={np.round(model.w, 4)}, b={model.b:.4f}")
print(f"Final BCE={model.loss_history[-1]:.4f}, train accuracy={acc * 100:.1f}%")

## 5. Decision boundary + loss curve

The black contour is where \(p = 0.5\) — the linear decision boundary in feature space.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

pad = 1.0
x_min, x_max = X[:, 0].min() - pad, X[:, 0].max() + pad
y_min, y_max = X[:, 1].min() - pad, X[:, 1].max() + pad
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = np.column_stack([xx.ravel(), yy.ravel()])
zz = model.predict_proba(grid).reshape(xx.shape)

axes[0].contourf(xx, yy, zz, levels=20, cmap="RdBu", alpha=0.55)
axes[0].contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)
axes[0].scatter(X[y == 0, 0], X[y == 0, 1], c="C0", edgecolors="k", linewidths=0.4, label="class 0", alpha=0.85)
axes[0].scatter(X[y == 1, 0], X[y == 1, 1], c="C3", edgecolors="k", linewidths=0.4, label="class 1", alpha=0.85)
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")
axes[0].set_title("Decision boundary (p = 0.5 contour)")
axes[0].legend(loc="upper left")

axes[1].plot(model.loss_history, color="C0")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("BCE loss")
axes[1].set_title("Training loss (gradient descent)")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Takeaways

- Logistic regression = linear score + sigmoid → class probability.
- BCE pushes probabilities toward the true labels; wrong *confident* predictions cost more.
- Gradients of BCE + sigmoid simplify to \((p - y)\) — elegant and efficient.
- In 2D, the decision boundary is a straight line (hyperplane in higher dimensions).
- Next week ideas: multi-class (softmax), regularization, or a tiny neural net from scratch.